# PV forecasting with a TCN (24h ahead) — time-series k-fold
This notebook trains a causal Temporal Convolutional Network (TCN) to predict the next **24 hourly PV values** from **weather + time features**.

- **No PV lag** is used as input.
- **Scaling** is fit on the training split only.
- **TimeSeriesSplit** provides time-aware k-fold validation on the training data.


In [ ]:

# --- Imports ---
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.parametrizations import spectral_norm

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## Data
Provide a merged dataframe `df_merged` with hourly timestamps and columns:
- `timestamp` (datetime)
- `pv` (target)
- weather features

If you already have `df_merged`, you can skip the merge cell.

In [ ]:

# If you already have df_merged, you can skip this cell.
# Example merge (edit column names as needed):

def build_df_merged(df_pv: pd.DataFrame, df_weather: pd.DataFrame,
                    pv_time_col="TimestampInUtc", pv_col="pv",
                    weather_time_col="timestamp") -> pd.DataFrame:
    df_pv = df_pv.copy()
    df_weather = df_weather.copy()

    df_pv[pv_time_col] = pd.to_datetime(df_pv[pv_time_col]).dt.floor("H")
    df_weather[weather_time_col] = pd.to_datetime(df_weather[weather_time_col]).dt.floor("H")

    df_pv = df_pv.rename(columns={pv_time_col: "timestamp"})
    if pv_col != "pv":
        df_pv = df_pv.rename(columns={pv_col: "pv"})

    df_weather = df_weather.rename(columns={weather_time_col: "timestamp"})

    df = (pd.merge(df_weather, df_pv[["timestamp", "pv"]], on="timestamp", how="inner")
            .sort_values("timestamp")
            .reset_index(drop=True))

    # Ensure hourly regularity & fill gaps (interpolate then ffill/bfill)
    df = df.set_index("timestamp").asfreq("H")
    df = df.interpolate(method="time").ffill().bfill()
    df = df.reset_index()

    return df

# Example usage:
# df_merged = build_df_merged(df_pv, df_weather, pv_time_col="TimestampInUtc", pv_col="pv", weather_time_col="timestamp")

df_merged.head()


## Feature engineering
We add cyclic time encodings. Recommended for PV:
- hour of day (sin/cos)
- day of year (sin/cos)

You can keep month encodings too, but day-of-year usually makes month redundant.

In [ ]:

# Ensure timestamp dtype
df_merged["timestamp"] = pd.to_datetime(df_merged["timestamp"])

# Basic time fields
df_merged["hour"] = df_merged["timestamp"].dt.hour
df_merged["day_of_year"] = df_merged["timestamp"].dt.dayofyear

# Cyclic encodings
df_merged["hour_sin"] = np.sin(2*np.pi*df_merged["hour"]/24)
df_merged["hour_cos"] = np.cos(2*np.pi*df_merged["hour"]/24)

df_merged["doy_sin"]  = np.sin(2*np.pi*df_merged["day_of_year"]/365.25)
df_merged["doy_cos"]  = np.cos(2*np.pi*df_merged["day_of_year"]/365.25)

df_merged.head()


## Train/test split
We hold out the last 1752 hours (~73 days) as a fixed test set.

In [ ]:

TEST_HOURS = 1752

df_test  = df_merged.iloc[-TEST_HOURS:].copy()
df_train = df_merged.iloc[:-TEST_HOURS].copy()

print("train:", df_train.shape, "test:", df_test.shape)
print("train range:", df_train["timestamp"].min(), "→", df_train["timestamp"].max())
print("test  range:", df_test["timestamp"].min(),  "→", df_test["timestamp"].max())


## Column groups
- Continuous weather features are standardized.
- Cyclic sin/cos features are left as-is.
- Target `pv` is standardized (recommended for stable training).

In [ ]:

TARGET_COL = "pv"

CONTINUOUS_FEATURES = [
    "Temperature",
    "Sunshine Duration",
    "Shortwave Radiation",
    "Direct Shortwave Radiation",
    "Diffuse Shortwave Radiation",
    "Snowfall Amount",
    "Relative Humidity",
    "Cloud Cover Total",
]

CYCLIC_FEATURES = [
    "hour_sin","hour_cos",
    "doy_sin","doy_cos",
]

# Safety: drop missing columns automatically (helps when your weather set differs)
CONTINUOUS_FEATURES = [c for c in CONTINUOUS_FEATURES if c in df_train.columns]
CYCLIC_FEATURES     = [c for c in CYCLIC_FEATURES if c in df_train.columns]

print("Continuous:", CONTINUOUS_FEATURES)
print("Cyclic:", CYCLIC_FEATURES)


In [ ]:

def prepare_scaled_arrays(df_train: pd.DataFrame, df_test: pd.DataFrame):
    # Continuous
    Xc_tr = df_train[CONTINUOUS_FEATURES].values
    Xc_te = df_test[CONTINUOUS_FEATURES].values

    # Cyclic (already in [-1,1])
    Xcy_tr = df_train[CYCLIC_FEATURES].values
    Xcy_te = df_test[CYCLIC_FEATURES].values

    # Target
    y_tr = df_train[[TARGET_COL]].values
    y_te = df_test[[TARGET_COL]].values

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    Xc_tr_s = scaler_X.fit_transform(Xc_tr)
    Xc_te_s = scaler_X.transform(Xc_te)

    y_tr_s = scaler_y.fit_transform(y_tr)
    y_te_s = scaler_y.transform(y_te)

    X_tr = np.concatenate([Xc_tr_s, Xcy_tr], axis=1).astype(np.float32)
    X_te = np.concatenate([Xc_te_s, Xcy_te], axis=1).astype(np.float32)

    y_tr = y_tr_s.astype(np.float32).ravel()   # (N,)
    y_te = y_te_s.astype(np.float32).ravel()

    return X_tr, y_tr, X_te, y_te, scaler_X, scaler_y

X_train, y_train, X_test, y_test, scaler_X, scaler_y = prepare_scaled_arrays(df_train, df_test)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)


## Sequence building (24h ahead)
We predict the **next 24 PV values**.
- input: `lookback` hours of features
- output: `horizon=24` PV values

In [ ]:

LOOKBACK = 168   # tune: 24, 48, 72, 168
HORIZON  = 24

def create_sequences_multistep(X, y, lookback, horizon):
    Xs, ys = [], []
    last_i = len(X) - lookback - horizon + 1
    for i in range(last_i):
        Xs.append(X[i:i+lookback])
        ys.append(y[i+lookback:i+lookback+horizon])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_train_seq, y_train_seq = create_sequences_multistep(X_train, y_train, LOOKBACK, HORIZON)
X_test_seq,  y_test_seq  = create_sequences_multistep(X_test,  y_test,  LOOKBACK, HORIZON)

print("Train seq:", X_train_seq.shape, y_train_seq.shape)
print("Test  seq:", X_test_seq.shape,  y_test_seq.shape)


In [ ]:

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)  # (N, L, F)
        self.y = torch.from_numpy(y)  # (N, H)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


## Model: Causal TCN

In [ ]:

class Chomp1d(nn.Module):
    """Remove extra padding from causal conv to keep output length same as input"""
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, dilation, padding, dropout, causal=True):
        super().__init__()
        self.conv1 = spectral_norm(nn.Conv1d(in_channels, out_channels, kernel_size,
                                             stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding) if causal else nn.Identity()
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = spectral_norm(nn.Conv1d(out_channels, out_channels, kernel_size,
                                             stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding) if causal else nn.Identity()
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.drop1,
                                 self.conv2, self.chomp2, self.relu2, self.drop2)

        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = self.downsample(x)
        return self.relu(out + res)

class TCN(nn.Module):
    def __init__(self, input_size, output_size, num_channels, kernel_size=3, dropout=0.2, causal=True):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation = 2 ** i
            in_ch = input_size if i == 0 else num_channels[i-1]
            out_ch = num_channels[i]
            padding = (kernel_size - 1) * dilation
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size, stride=1, dilation=dilation,
                                        padding=padding, dropout=dropout, causal=causal))
        self.tcn = nn.Sequential(*layers)
        self.head = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        # x: (B, L, F) -> conv wants (B, F, L)
        x = x.transpose(1, 2)
        y = self.tcn(x)               # (B, C, L)
        y_last = y[:, :, -1]          # last time step (B, C)
        out = self.head(y_last)       # (B, HORIZON)
        return out


## Training utilities

In [ ]:

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total, n = 0.0, 0
    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total += loss.item() * Xb.size(0)
        n += Xb.size(0)
    return total / n

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    total, n = 0.0, 0
    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)
        pred = model(Xb)
        loss = criterion(pred, yb)
        total += loss.item() * Xb.size(0)
        n += Xb.size(0)
    return total / n


## Time-series aware k-fold on training set
We run `TimeSeriesSplit` over the **training sequences** (no shuffling).

In [ ]:

# Hyperparams
BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-3

NUM_CHANNELS = [32, 32, 32, 32]
KERNEL_SIZE = 3
DROPOUT = 0.2
CAUSAL = True

criterion = nn.MSELoss()

def run_timeseries_kfold(X_seq, y_seq, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)

    fold_losses = []
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_seq)):
        train_ds = SeqDataset(X_seq[tr_idx], y_seq[tr_idx])
        val_ds   = SeqDataset(X_seq[va_idx], y_seq[va_idx])

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        model = TCN(
            input_size=X_seq.shape[-1],
            output_size=y_seq.shape[-1],
            num_channels=NUM_CHANNELS,
            kernel_size=KERNEL_SIZE,
            dropout=DROPOUT,
            causal=CAUSAL
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        best_val = float("inf")
        best_state = None

        for epoch in range(EPOCHS):
            tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            va_loss = eval_one_epoch(model, val_loader, criterion)

            if va_loss < best_val:
                best_val = va_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        fold_losses.append(best_val)
        print(f"Fold {fold+1}/{n_splits} | best val MSE: {best_val:.5f}")

    return fold_losses

fold_losses = run_timeseries_kfold(X_train_seq, y_train_seq, n_splits=5)
print("Fold losses:", fold_losses)
print("Mean ± std:", np.mean(fold_losses), "±", np.std(fold_losses))


## Train final model on full training set, evaluate on fixed test set

In [ ]:

# Final training on full train sequences
train_loader = DataLoader(SeqDataset(X_train_seq, y_train_seq), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(SeqDataset(X_test_seq,  y_test_seq),  batch_size=BATCH_SIZE, shuffle=False)

model = TCN(
    input_size=X_train_seq.shape[-1],
    output_size=y_train_seq.shape[-1],
    num_channels=NUM_CHANNELS,
    kernel_size=KERNEL_SIZE,
    dropout=DROPOUT,
    causal=CAUSAL
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_train = float("inf")
best_state = None

for epoch in range(EPOCHS):
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    if tr_loss < best_train:
        best_train = tr_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"Epoch {epoch+1:02d} | train MSE: {tr_loss:.5f}")

# Load best train state (simple)
model.load_state_dict(best_state)

test_mse = eval_one_epoch(model, test_loader, criterion)
print("Test MSE (scaled PV):", test_mse)


## Example: convert one prediction back to PV units

In [ ]:

@torch.no_grad()
def predict_one_batch(model, loader):
    model.eval()
    Xb, yb = next(iter(loader))
    Xb = Xb.to(device)
    pred = model(Xb).cpu().numpy()
    y_true = yb.numpy()
    return pred, y_true

pred_scaled, y_true_scaled = predict_one_batch(model, test_loader)

# Inverse transform to real PV units
pred_real = scaler_y.inverse_transform(pred_scaled.reshape(-1,1)).reshape(pred_scaled.shape)
true_real = scaler_y.inverse_transform(y_true_scaled.reshape(-1,1)).reshape(y_true_scaled.shape)

print("pred_real shape:", pred_real.shape)
print("First sample, first 5 hours (pred vs true):")
print(np.stack([pred_real[0,:5], true_real[0,:5]], axis=1))
